# Phase 1: Multi-prototype study

## Section 1 - Purpose and overview

This study asks whether representing a class with more than one prototype corrects the known limits of perception: the empty mouth slot predicted as mouth, the loose mouth seen from the side, and the identity of the two arms, without making the rest worse.

The current method represents each class with one prototype, the normalized mean of its annotated patches (Snell et al. 2017, Sect. 2.2). A diagnostic on the reference frames showed why a mean can fail. The empty mouth slot is annotated, inside the body polygon of 3_disassembled_ordered_f0001, but it is one patch out of 251 body patches, and that patch is closer to the mouth prototype than to the body prototype. An example can be present and still be lost in the mean.

Two changes are made, in this order: first the examples, then how they are used.

The examples follow a coverage rule: for each piece, the support set contains at least one example of each state in which the piece appears during the assembly. This is our own design rule. It is in the spirit of the viewpoint annotation of PASCAL VOC, introduced for methods that treat different viewpoints separately (Everingham et al. 2010, Sect. 3.3). Two states are missing from the current support set, and one frame is added for each, from reference recordings only. caos_f0043 adds the loose mouth seen from the side (use_labels: mouth). 4_partially_assembled_f0002 adds the body on the base with the face not yet complete, including the empty mouth slot, and a loose mouth seen at three quarters (use_labels: body, mouth). All visible pieces are annotated in both frames, so that unused pieces become ignored patches and not background. 4_partially_assembled_f0002 was a candidate in the support-set study and was not selected there; it enters now because of the coverage rule, not because of a score. The support set of 10 frames is adopted by rule, and its comparison with the support set of 8 frames is reported only as a description.

How the examples are used is the question of the study. Mensink et al. (2013, Sect. 1) contrast two distance-based classifiers: the nearest class mean, which represents a class by its mean, and the k-nearest neighbour, which uses all its examples. They also extend the first to several centroids per class (Sect. 3.3). Three variants follow from this. V0 is the current method, one prototype per class. V1 has one prototype per class and aspect, mounted or loose, and background and hand keep a single prototype. The aspect of each piece in each reference frame is declared as an annotated attribute, as the viewpoint in PASCAL VOC (Everingham et al. 2010, Sect. 3.3), rather than computed: a geometric rule would need a distance to the figure, and the labelled patches of a mounted arm, mostly the hand, lie several patches away from the body. This is the idea of several centroids per class, with the groups given by the aspect rather than by k-means, because the slot and the side view of the mouth cover one to three patches and could not form a cluster of their own. On unit vectors it corresponds to a mixture of von Mises-Fisher distributions (Banerjee et al. 2005, Sect. 3). V2 uses every annotated patch as a prototype, the nearest neighbour. In all variants the similarity of a class is the highest similarity among its prototypes, and the background margin is applied as before.

The hypotheses are written before running. In V0 the two new frames change little, because their few patches are lost in the means, as in the support-set study. V2 should predict the empty slot as body, because the annotated slot patches become body prototypes; the risk is that body patches next to the face, which already look like face pieces, take patches from the face pieces. V1 probably does not correct the slot, which stays a few patches in the mean of its aspect. The side view of the mouth keeps one patch after labelling, so only V2 can use it. No variant should correct the identity of the arms along their stick, where the patches of the two arms look the same. Mensink et al. found the nearest class mean better than the nearest neighbour in their setting, so V2 may also be worse overall.

The decision rule is the one of the support-set study. A variant replaces V0 only if the paired 95% bootstrap interval of the mIoU difference, with the arms merged and background margin 0.03, lies entirely above zero (Koehn 2004, Sect. 6). If both variants pass, V1 is chosen, being simpler and faster, unless V2 is better than V1 by the same criterion. Three targeted measures are reported but do not enter the rule: body patches predicted as mouth in the frames where the mouth is not mounted, the share of arm patches given the right arm, loose and mounted, and the mouth IoU in the two validation frames of 1_disassembled_controlled_caos. The time of classification is also measured, since V2 compares each patch with tens of thousands of prototypes.

The two new frames were motivated by errors seen on validation, and the variants are chosen on validation, which makes the results optimistic (Cawley and Talbot 2010, Sect. 4.4 and 5). For this reason the variants are few and fixed here, and the test sequence (assembly_1) is not read.

Section 2 sets the configuration. Section 3 checks that the new code reproduces the current results. Section 4 builds the variants, Section 5 compares them, Section 6 reports the targeted measures, Section 7 the time, and Section 8 applies the rule.

## Section 2 - Configuration

All paths, frames and hyperparameters in one place. The values are those of the pipeline (Section 2.5) and of the support-set study (Section 2.4). The support set has two more frames, added by the coverage rule (Section 1).

In [13]:
# 2.1 Dependencies: install once per environment with pip install -r requirements.txt (README, Setup)

In [ ]:
# 2.2 Imports
%load_ext autoreload
%autoreload 2

import sys
import json
import time
import hashlib
import platform
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image
from scipy import ndimage
import transformers
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cv2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [15]:
# 2.3 Paths and shared module
PROJECT_ROOT = Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))   # makes src/perception importable from notebooks/

MODEL_DIR = PROJECT_ROOT / "models" / "c_radio_weights"
FRAMES_DIR = PROJECT_ROOT / "data" / "extracted_frames"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "phase_1_multi_prototype_study"
PIPELINE_DIR = PROJECT_ROOT / "outputs" / "phase_1_perception"
PIPELINE_PROTOTYPES_PATH = PIPELINE_DIR / "prototypes.npz"                    # saved by the pipeline (v1.3): 8 reference frames
PIPELINE_SIMILARITIES_PATH = PIPELINE_DIR / "validation_similarities.npz"     # saved by the pipeline (v1.3): 8 validation frames

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for name, path in {"MODEL_DIR": MODEL_DIR, "FRAMES_DIR": FRAMES_DIR, "SRC_DIR": SRC_DIR,
                   "PIPELINE_PROTOTYPES_PATH": PIPELINE_PROTOTYPES_PATH,
                   "PIPELINE_SIMILARITIES_PATH": PIPELINE_SIMILARITIES_PATH}.items():
    status = "OK" if path.exists() else "MISSING"
    print(f"[{status}] {name}: {path}")

# Shared code (src/perception)
from perception.classes import (CLASS_NAMES, BACKGROUND_ID, HAND_LABEL, PIECE_LABELS, IGNORE_ID,
                                CLASS_COLORS, PATCH_LABEL_NAMES)
from perception.features import FeatureExtractor
from perception.labels import sequence_of, check_annotation, rasterize_annotation, patch_label_grid
from perception.prototypes import (build_prototypes, stack_prototypes, load_prototypes, classify_patches,
                                   assign_labels, group_prototypes, patch_prototypes, class_similarities)
from perception.metrics import (confusion_matrix, iou_from_confusions, mean_iou, ignore_boundary_ring,
                                MERGED_NAMES, MERGED_PIECES, merge_arms, merged_ids,
                                frame_iou, frame_miou, bootstrap_miou, bootstrap_class_iou, percentile_interval)
from perception.visualization import label_grid_to_rgb

[OK] MODEL_DIR: c:\Users\ADV\Universita\Magistrale\Tesi\vad-project\models\c_radio_weights
[OK] FRAMES_DIR: c:\Users\ADV\Universita\Magistrale\Tesi\vad-project\data\extracted_frames
[OK] SRC_DIR: c:\Users\ADV\Universita\Magistrale\Tesi\vad-project\src
[OK] PIPELINE_PROTOTYPES_PATH: c:\Users\ADV\Universita\Magistrale\Tesi\vad-project\outputs\phase_1_perception\prototypes.npz
[OK] PIPELINE_SIMILARITIES_PATH: c:\Users\ADV\Universita\Magistrale\Tesi\vad-project\outputs\phase_1_perception\validation_similarities.npz


In [16]:
# 2.4 Hyperparameters
# Same values as the pipeline (Section 2.5) and the support-set study (Section 2.4).

# --- Feature extraction ---
DEVICE = "cuda"
DTYPE = torch.bfloat16          # input resolution: native 1024x1024 (resolution study)

# --- Patch labelling ---
MIN_LABEL_COVERAGE = 0.5        # same rule as the pipeline (Section 5)

# --- Classification ---
CONFIDENCE_MARGIN = 0.03        # background margin, provisional (pipeline, Section 9): not tuned here
EVAL_MARGINS = [0.0, CONFIDENCE_MARGIN]
PROTOTYPE_CHUNK = 4096          # prototypes compared at once in class_similarities: memory only, results unchanged

# --- Evaluation ---
N_BOOTSTRAP = 1000              # resamples of the validation frames (support-set study, Section 5.4)
BOUNDARY_RING = 1               # secondary metric: background patches within 1 patch of an object are ignored

# --- Latency (Section 7), same protocol as the pipeline, Section 12 ---
N_WARMUP, N_RUNS = 5, 50

# --- Reproducibility ---
RANDOM_SEED = 0
np.random.seed(RANDOM_SEED)
_ = torch.manual_seed(RANDOM_SEED)

In [17]:
# 2.5 Frames and split
# Validation: errors are inspected and the variants are chosen here.
# Test: held out until the end of all studies. This notebook never reads a test frame.
VALIDATION_SEQUENCES = {"assembly_2", "1_disassembled_controlled_caos"}
TEST_SEQUENCES = {"assembly_1"}
assert not VALIDATION_SEQUENCES & TEST_SEQUENCES


def frame_path(frame, suffix=".jpg"):
    """Path of a frame or of its annotation. Refuses test frames, so the test set cannot be read by mistake."""
    assert sequence_of(frame) not in TEST_SEQUENCES, f"{frame} belongs to the held-out test set"
    return FRAMES_DIR / f"{frame}{suffix}"


# Current support set (support-set study, Section 6.5), in the order of the pipeline (Section 4.1)
CURRENT_REFERENCE_FRAMES = [
    {"frame": "5_assembled_f0002",            "role": "assembled",                   "use_labels": PIECE_LABELS},
    {"frame": "caos_f0149",                   "role": "partially assembled",         "use_labels": PIECE_LABELS},
    {"frame": "5_assembled_f0019",            "role": "assembled, 3/4 view",         "use_labels": PIECE_LABELS},
    {"frame": "3_disassembled_ordered_f0001", "role": "disassembled",                "use_labels": PIECE_LABELS - {"base"}},
    {"frame": "2_disassembled_caos_f0016",    "role": "disassembled, top-down view", "use_labels": PIECE_LABELS - {"base"}},
    {"frame": "Take_3_1_Base_f0099",          "role": "base only, upright",          "use_labels": {"base"}},
    {"frame": "Take_3_1_Base_f0131",          "role": "base only, lying",            "use_labels": {"base"}},
    {"frame": "caos_f0211",                   "role": "hands",                       "use_labels": {HAND_LABEL}},
]

# Frames added by the coverage rule (Section 1): one per missing state
COVERAGE_FRAMES = [
    {"frame": "4_partially_assembled_f0002",  "role": "body on base, face incomplete (empty mouth slot); mouth loose at 3/4",
     "use_labels": {"body", "mouth"}},
    {"frame": "caos_f0043",                   "role": "mouth loose, side view", "use_labels": {"mouth"}},
]
REFERENCE_FRAMES = CURRENT_REFERENCE_FRAMES + COVERAGE_FRAMES

# Validation frames in the order of the support-set study (Section 2.5): bootstrap resamples pick frames by position
VALIDATION_FRAMES = [
    {"frame": "assembly_2_f0003", "stage": "all pieces loose"},
    {"frame": "assembly_2_f0017", "stage": "body on base"},
    {"frame": "assembly_2_f0030", "stage": "hands: piece held in hand"},
    {"frame": "assembly_2_f0046", "stage": "face (eyes, nose, moustache, mouth)"},
    {"frame": "assembly_2_f0057", "stage": "ears mounted"},
    {"frame": "assembly_2_f0068", "stage": "arms mounted"},
    {"frame": "1_disassembled_controlled_caos_f0001", "stage": "disassembled, front view", "exclude": {"base"}},
    {"frame": "1_disassembled_controlled_caos_f0016", "stage": "disassembled, side view",  "exclude": {"base"}},
]

# No leakage: reference frames never come from validation or test sequences
for ref in REFERENCE_FRAMES:
    assert sequence_of(ref["frame"]) not in VALIDATION_SEQUENCES | TEST_SEQUENCES, f"Leakage: {ref['frame']}"
    ref["image_path"], ref["json_path"] = frame_path(ref["frame"]), frame_path(ref["frame"], ".json")
for val in VALIDATION_FRAMES:
    assert sequence_of(val["frame"]) in VALIDATION_SEQUENCES, f"{val['frame']} is not a validation frame"
    assert frame_path(val["frame"]).exists(), f"Missing frame: {val['frame']}"

GT_LABELS = PIECE_LABELS | {HAND_LABEL}   # ground truth: every official class; legacy labels become ignored patches

print(f"{len(REFERENCE_FRAMES)} reference frames ({len(CURRENT_REFERENCE_FRAMES)} current + "
      f"{len(COVERAGE_FRAMES)} added by coverage), {len(VALIDATION_FRAMES)} validation frames")

10 reference frames (8 current + 2 added by coverage), 8 validation frames


In [18]:
# 2.5b Aspect of the pieces in each reference frame: an annotated attribute, as the viewpoint in PASCAL VOC
# (Everingham et al. 2010, Sect. 3.3). Mounted means: body on the base, base under the body, any other piece
# inserted in the body.
# "mounted": every piece is mounted; "loose": every piece is loose;
# "mixed": the mounted pieces are the polygons with group_id 1 in the labelme file;
# None: a single aspect (frames with hands only).
FRAME_ASPECT = {
    "5_assembled_f0002": "mounted",
    "caos_f0149": "mixed",
    "5_assembled_f0019": "mounted",
    "3_disassembled_ordered_f0001": "loose",
    "2_disassembled_caos_f0016": "loose",
    "Take_3_1_Base_f0099": "loose",          # base alone counts as not mounted
    "Take_3_1_Base_f0131": "loose",
    "caos_f0211": None,
    "4_partially_assembled_f0002": "mixed",
    "caos_f0043": "loose",
}
assert set(FRAME_ASPECT) == {ref["frame"] for ref in REFERENCE_FRAMES}, "Every reference frame needs an aspect"

In [19]:
# 2.6 Annotation fingerprints: data/ is not tracked by git, so the notebook records which annotations it used
def fingerprint(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()[:12]

display(pd.DataFrame({
    "set": ["reference"] * len(REFERENCE_FRAMES) + ["validation"] * len(VALIDATION_FRAMES),
    "annotation sha256": [fingerprint(ref["json_path"]) for ref in REFERENCE_FRAMES]
                         + [fingerprint(frame_path(val["frame"], ".json")) for val in VALIDATION_FRAMES],
}, index=[ref["frame"] for ref in REFERENCE_FRAMES] + [val["frame"] for val in VALIDATION_FRAMES]))

,set,annotation sha256
5_assembled_f0002,reference,e85d1c179582
caos_f0149,reference,651709826565
5_assembled_f0019,reference,1cd8cdcc2c9b
3_disassembled_ordered_f0001,reference,319c3601f0a1
2_disassembled_caos_f0016,reference,f17ab6af8218
Take_3_1_Base_f0099,reference,853e60024c17
Take_3_1_Base_f0131,reference,f07f67aae3a9
caos_f0211,reference,3c56b4804e54
4_partially_assembled_f0002,reference,32a94339f2b4
caos_f0043,reference,3b555dd93c2c


In [20]:
# 2.7 Environment
print(f"Python       : {platform.python_version()}")
print(f"PyTorch      : {torch.__version__}")
print(f"Transformers : {transformers.__version__}")
print(f"CUDA         : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU          : {torch.cuda.get_device_name(0)}")

Python       : 3.10.0
PyTorch      : 2.5.1+cu121
Transformers : 5.17.0
CUDA         : True
GPU          : NVIDIA GeForce RTX 4060 Laptop GPU


------------------------------------------------------------------------------------------------------------------------------------------

References Section 2: none. The values are taken from the pipeline (Section 2.5) and the support-set study (Section 2.4); the two added frames follow the coverage rule of Section 1.

------------------------------------------------------------------------------------------------------------------------------------------

## Section 3 - Regression check

The prototypes can now be built and compared with new shared functions (group_prototypes, patch_prototypes, class_similarities). Before using them for the variants, this section checks that in the simplest case, one prototype per class from the current 8 reference frames, they reproduce the pipeline exactly. Otherwise a difference between the variants could come from the code and not from the method.

Four things are checked. The prototypes are bitwise identical to those saved by the pipeline (v1.3). group_prototypes, with a single group per class, gives the same prototypes as build_prototypes. class_similarities, with one prototype per class, gives the validation similarities saved by the pipeline, bitwise. The mIoU of the current method is the one reported by the support-set study. Exact comparisons are possible because feature extraction is deterministic on this machine (support-set study, Section 3.2), although this is not guaranteed across releases or platforms (PyTorch documentation, Reproducibility).

This section also extracts the features of all 10 reference frames and of the validation frames, and prepares the ground truth, as in the support-set study (Section 4.2).

In [21]:
# 3.1 Loading the model (shared FeatureExtractor, same settings as the pipeline)
extractor = FeatureExtractor(MODEL_DIR, device=DEVICE, dtype=DTYPE)
PATCH_SIZE = extractor.patch_size
print(f"Model loaded, patch size {PATCH_SIZE}x{PATCH_SIZE}")

Loading weights:   0%|          | 0/330 [00:00<?, ?it/s]

Model loaded, patch size 16x16


In [22]:
# 3.2 Reference frames: features and patch labels; V0 from the current 8 frames, compared with the pipeline file
for ref in REFERENCE_FRAMES:
    check_annotation(ref["image_path"], ref["json_path"])
    image = Image.open(ref["image_path"]).convert("RGB")
    ref["features"], size = extractor.extract(image)
    # patches that enter the prototypes (use_labels)
    ref["patch_labels"], _ = patch_label_grid(rasterize_annotation(ref["json_path"], ref["use_labels"], size),
                                              PATCH_SIZE, MIN_LABEL_COVERAGE)
    # every annotated piece: used only to decide whether a piece is mounted or loose (Section 4), never as prototype
    ref["all_patch_labels"], _ = patch_label_grid(rasterize_annotation(ref["json_path"], GT_LABELS, size),
                                                  PATCH_SIZE, MIN_LABEL_COVERAGE)

prototypes_v0, counts_v0 = build_prototypes(CURRENT_REFERENCE_FRAMES)
matrix_v0 = stack_prototypes(prototypes_v0)
owners_v0 = np.arange(len(CLASS_NAMES))          # one prototype per class, in class order

pipeline = load_prototypes(PIPELINE_PROTOTYPES_PATH)
assert pipeline["class_names"] == CLASS_NAMES, "Class order differs from the pipeline file"
print("V0 prototypes bitwise identical to the pipeline:", np.array_equal(matrix_v0, pipeline["prototypes"]))

# group_prototypes with a single group per class must reproduce build_prototypes
single_group = [{**ref, "patch_groups": np.zeros_like(ref["patch_labels"])} for ref in CURRENT_REFERENCE_FRAMES]
grouped, grouped_owners, _, _ = group_prototypes(single_group)
print("group_prototypes (one group) identical to build_prototypes:",
      np.array_equal(grouped, matrix_v0) and np.array_equal(grouped_owners, owners_v0))

V0 prototypes bitwise identical to the pipeline: True
group_prototypes (one group) identical to build_prototypes: True


Observation 3.2: Built from the current 8 reference frames, the V0 prototypes are bitwise identical to those saved by the pipeline. group_prototypes with a single group per class gives exactly the same prototypes as build_prototypes.

In [23]:
# 3.3 Validation frames: ground truth (support-set study, Section 4.2), features, and similarities with V0
# computed by class_similarities, compared with the file saved by the pipeline (stored by frame name)
with np.load(PIPELINE_SIMILARITIES_PATH) as data:
    pipeline_similarities = {key: data[key] for key in data.files if key.endswith("__similarities")}

identical = []
for val in VALIDATION_FRAMES:
    image_path, json_path = frame_path(val["frame"]), frame_path(val["frame"], ".json")
    check_annotation(image_path, json_path)
    image = Image.open(image_path).convert("RGB")
    val["features"], size = extractor.extract(image)
    val["gt_labels"], _ = patch_label_grid(rasterize_annotation(json_path, GT_LABELS, size), PATCH_SIZE, MIN_LABEL_COVERAGE)
    val["gt_tolerant"] = ignore_boundary_ring(val["gt_labels"], BOUNDARY_RING)
    val["excluded_ids"] = {CLASS_NAMES.index(name) for name in val.get("exclude", set())}
    similarities = class_similarities(val["features"], matrix_v0, owners_v0, chunk=PROTOTYPE_CHUNK)
    identical.append(np.array_equal(similarities, pipeline_similarities[f"{val['frame']}__similarities"]))

EXCLUDED = [val["excluded_ids"] for val in VALIDATION_FRAMES]
print(f"class_similarities bitwise identical to the pipeline on {sum(identical)} of {len(identical)} validation frames")

class_similarities bitwise identical to the pipeline on 8 of 8 validation frames


Observation 3.3: With one prototype per class, class_similarities gives the validation similarities saved by the pipeline, bitwise, on all 8 validation frames.

In [24]:
# 3.4 Evaluation for any variant (prototype matrix + class of each row) and bootstrap resamples
def predict(features, prototypes, owners, margin):
    """Patch labels: highest similarity among the prototypes of each class, then the background margin."""
    return assign_labels(class_similarities(features, prototypes, owners, chunk=PROTOTYPE_CHUNK), margin)


def evaluate(prototypes, owners, margin, gt_key="gt_labels"):
    """Per-frame confusion matrices on the validation frames. gt_key: "gt_labels" or "gt_tolerant"."""
    return [confusion_matrix(val[gt_key], predict(val["features"], prototypes, owners, margin))
            for val in VALIDATION_FRAMES]


rng = np.random.default_rng(RANDOM_SEED)
BOOTSTRAP_INDICES = rng.integers(0, len(VALIDATION_FRAMES), size=(N_BOOTSTRAP, len(VALIDATION_FRAMES)))

conf_v0 = {gt_key: evaluate(matrix_v0, owners_v0, CONFIDENCE_MARGIN, gt_key) for gt_key in ["gt_labels", "gt_tolerant"]}
display(pd.DataFrame({
    "obtained": [frame_miou(conf_v0["gt_labels"], EXCLUDED), frame_miou(conf_v0["gt_labels"], EXCLUDED, merge=True),
                 frame_miou(conf_v0["gt_tolerant"], EXCLUDED), frame_miou(conf_v0["gt_tolerant"], EXCLUDED, merge=True)],
    "support-set study": [0.658, 0.709, 0.737, 0.805],
}, index=["10 pieces, main metric", "arms merged, main metric",
          "10 pieces, boundary ring ignored", "arms merged, boundary ring ignored"]).round(3))

,obtained,support-set study
"10 pieces, main metric",0.658,0.658
"arms merged, main metric",0.709,0.709
"10 pieces, boundary ring ignored",0.737,0.737
"arms merged, boundary ring ignored",0.805,0.805


------------------------------------------------------------------------------------------------------------------------------------------

Observation 3.4: The current method scores the mIoU reported by the support-set study at background margin 0.03: 0.658 over 10 pieces and 0.709 with the arms merged on the main metric, 0.737 and 0.805 with the boundary ring ignored. The new code reproduces the current method exactly, so differences between the variants in the next sections come from the variants.

References Section 3: PyTorch documentation, Reproducibility (docs.pytorch.org/docs/stable/notes/randomness.html): completely reproducible results are not guaranteed across releases, commits or platforms.

------------------------------------------------------------------------------------------------------------------------------------------

## Section 4 - Aspects and variants

This section assigns an aspect, mounted or loose, to every labelled patch of the reference frames, and builds the three variants on the support set of 10 frames.

The aspect is an annotated attribute, as the viewpoint in PASCAL VOC (Everingham et al. 2010, Sect. 3.3). In most reference frames all pieces share one state, declared for the whole frame. In the two frames where some pieces are mounted and others are loose, the mounted polygons are marked in the annotation (group_id 1). A patch takes the aspect of the mounted polygons if they cover at least half of it, the same coverage rule used for the patch labels (pipeline, Section 5). Background and hand keep a single aspect.

V0 is the current method on the 10 frames, one prototype per class (Snell et al. 2017, Sect. 2.2). V1 has one prototype per class and aspect, the idea of several centroids per class (Mensink et al. 2013, Sect. 3.3) with the groups given by the aspect. V2 uses every labelled patch as a prototype, the nearest neighbour (Mensink et al. 2013, Sect. 1). V0 on the current 8 frames is kept only to describe the effect of the two frames added by the coverage rule.

In [25]:
# 4.1 Aspect of every labelled patch: 0 = loose, 1 = mounted (background and hand: a single aspect)
LOOSE, MOUNTED = 0, 1
HAND_ID = CLASS_NAMES.index(HAND_LABEL)


def mounted_pixels(json_path):
    """Pixel mask of the polygons marked as mounted (group_id 1) in a labelme file."""
    with open(json_path) as f:
        annotation = json.load(f)
    mask = np.zeros((annotation["imageHeight"], annotation["imageWidth"]), dtype=np.uint8)
    for shape in annotation["shapes"]:
        if shape.get("group_id") == 1:
            cv2.fillPoly(mask, [np.round(np.array(shape["points"])).astype(np.int32)], 1)
    return mask.astype(bool)


def patch_fraction(mask, patch):
    """Fraction of each patch covered by a pixel mask."""
    h, w = mask.shape
    return mask[:h // patch * patch, :w // patch * patch].reshape(h // patch, patch, w // patch, patch).mean(axis=(1, 3))


for ref in REFERENCE_FRAMES:
    aspect = FRAME_ASPECT[ref["frame"]]
    groups = np.full(ref["patch_labels"].shape, LOOSE)
    if aspect == "mounted":
        groups[:] = MOUNTED
    elif aspect == "mixed":
        groups[patch_fraction(mounted_pixels(ref["json_path"]), PATCH_SIZE) >= MIN_LABEL_COVERAGE] = MOUNTED
    groups[np.isin(ref["patch_labels"], [BACKGROUND_ID, HAND_ID])] = LOOSE     # single aspect
    ref["patch_groups"] = groups

# Patches per class and aspect over the 10 reference frames
counts = {(name, aspect): sum(int(((ref["patch_labels"] == i) & (ref["patch_groups"] == g)).sum())
                              for ref in REFERENCE_FRAMES)
          for i, name in enumerate(CLASS_NAMES) for g, aspect in [(LOOSE, "loose"), (MOUNTED, "mounted")]}
display(pd.Series(counts).unstack().loc[CLASS_NAMES])

# Visual check of the two mixed frames: mounted patches green, loose patches orange
fig, axes = plt.subplots(1, 2, figsize=(14, 7))
for ax, name in zip(axes, [f for f, a in FRAME_ASPECT.items() if a == "mixed"]):
    ref = next(r for r in REFERENCE_FRAMES if r["frame"] == name)
    image = Image.open(ref["image_path"]).convert("RGB")
    is_piece = ~np.isin(ref["patch_labels"], [BACKGROUND_ID, HAND_ID, IGNORE_ID])
    overlay = np.zeros((*ref["patch_labels"].shape, 4))
    overlay[is_piece & (ref["patch_groups"] == MOUNTED)] = (0.1, 0.8, 0.1, 0.7)
    overlay[is_piece & (ref["patch_groups"] == LOOSE)] = (1.0, 0.5, 0.0, 0.7)
    ax.imshow(image)
    ax.imshow(overlay, extent=(0, image.width, image.height, 0), interpolation="nearest")
    ax.set_title(f"{name} (green: mounted, orange: loose; only use_labels)", fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

NameError: name 'cv2' is not defined

In [ ]:
# 4.2 The variants: (prototype matrix, class of each row)
prototypes_10, counts_10 = build_prototypes(REFERENCE_FRAMES)
v1_matrix, v1_owners, v1_groups, v1_counts = group_prototypes(REFERENCE_FRAMES)
v2_matrix, v2_owners = patch_prototypes(REFERENCE_FRAMES)

VARIANTS = {
    "V0 (8 frames)": (matrix_v0, owners_v0),                                    # descriptive comparison only
    "V0": (stack_prototypes(prototypes_10), np.arange(len(CLASS_NAMES))),
    "V1": (v1_matrix, v1_owners),
    "V2": (v2_matrix, v2_owners),
}
display(pd.DataFrame({name: {"prototypes": len(m)} for name, (m, o) in VARIANTS.items()}).T)

# V1: one prototype per class and aspect, with the number of patches behind each
display(pd.DataFrame({"class": [CLASS_NAMES[c] for c in v1_owners],
                      "aspect": ["mounted" if g == MOUNTED else "loose" for g in v1_groups],
                      "patches": v1_counts}))

------------------------------------------------------------------------------------------------------------------------------------------

References Section 4:

Everingham et al. 2010, PASCAL VOC, Sect. 3.3: viewpoint annotated as an attribute of each object, to support methods that treat viewpoints separately 4.1.

Pipeline, Section 5: a patch takes a label if it covers at least half of the patch 4.1.

Snell et al. 2017, Sect. 2.2; Mensink et al. 2013, Sect. 1 and Sect. 3.3 4.2.

------------------------------------------------------------------------------------------------------------------------------------------